# Grid methods: power flow vs the paper's graph model

The two arms of the grid-method comparison ([docs/experiments/grid-methods.md](../../docs/experiments/grid-methods.md)) on one scenario, **before any learning**: every participant follows the net-position baseline, which ignores fees. So the physics (the AC power flow, recorded in every arm) is the same in all arms, and every difference below is the grid method alone.

Arms:

| Arm | `grid.method` | capacity | `dso.voltage_drop` |
|---|---|---|---|
| `ac` | `powerflow` | – | `distance` (as now) |
| `ac_measured` | `powerflow` | – | `measured` |
| `graph` | `graph` | `paper` (1/length, total = sum of ratings) | `distance` |
| `graph_ratings` | `graph` | `ratings` | `distance` |

In [1]:
import sys
from pathlib import Path

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "core").is_dir())
sys.path.insert(0, str(ROOT))

import pandas as pd

from core.env import MarketEnv, net_position_policy

SCENARIO = ROOT / "config/scenarios/validate_market.json"
DAYS = [f"2016-07-{d:02d}" for d in range(1, 15)]
ARMS = {
    "ac": {},
    "ac_measured": {"dso": {"voltage_drop": "measured"}},
    "graph": {"grid": {"method": "graph"}},
    "graph_ratings": {"grid": {"method": "graph", "graph_capacity": "ratings"}},
}

## Play every test day in every arm

One row per trade: its fee parts, the loading its fees saw on its path (`c_fee`) and the measured loading of the same lines (`c_measured`).

In [2]:
def lines_of(market, trade):
    dso = market.dso.id
    if dso in (trade.buyer_id, trade.seller_id):
        other = trade.seller_id if trade.buyer_id == dso else trade.buyer_id
        return market._upstream(other).lines
    return market._path(trade.buyer_id, trade.seller_id).lines


def play(arm, overrides):
    env = MarketEnv({"scenario": str(SCENARIO), "days": DAYS, **overrides})
    policy = net_position_policy(env)
    trades, steps = [], []
    for day in DAYS:
        obs, _ = env.reset(options={"day": day})
        done = False
        while not done:
            obs, _, term, _, _ = env.step(policy(obs))
            done = term["__all__"]
            r = env.last_record
            fee = r.signal or r.grid
            steps.append(dict(arm=arm, day=day, step=r.step,
                              measured_max=r.grid.max_line_loading_percent,
                              fee_max=fee.max_line_loading_percent,
                              grid_losses=r.grid.losses_kwh, trade_losses=r.losses_kwh))
            for t in r.trades:
                lines = lines_of(env.market, t)
                load = lambda a: max((a.line_loading_percent.get(l, 0) for l in lines), default=0) / 100
                trades.append(dict(arm=arm, day=day, step=r.step, q=t.quantity, p=t.price,
                                   value=t.quantity * t.price, dso=env.market.dso.id in (t.buyer_id, t.seller_id),
                                   c_fee=min(load(fee), 1), c_measured=min(load(r.grid), 1),
                                   **{f"fee_{k}": getattr(t.fee_parts, k) for k in
                                      ("congestion", "thermal", "voltage", "transmission", "zone", "imbalance")}))
    return trades, steps


trades, steps = [], []
for arm, overrides in ARMS.items():
    t, s = play(arm, overrides)
    trades += t
    steps += s
trades, steps = pd.DataFrame(trades), pd.DataFrame(steps)
len(trades), len(steps)

(5868, 1344)

## Same physics in every arm

The baseline ignores fees, so the measured line loading must be identical across arms.

In [3]:
steps.pivot_table(index=["day", "step"], columns="arm", values="measured_max").nunique(axis=1).max()

np.int64(1)

## Fees per arm

Totals over the 14 days, and as a share of the value traded.

In [4]:
parts = [c for c in trades if c.startswith("fee_")]
fees = trades.groupby("arm")[parts + ["value"]].sum()
fees["total"] = fees[parts].sum(axis=1)
fees["total / value"] = fees["total"] / fees["value"]
fees.round(3)

,fee_congestion,fee_thermal,fee_voltage,fee_transmission,fee_zone,fee_imbalance,value,total,total / value
arm,,,,,,,,,
ac,0.0,0.217,24.479,4.896,-11.343,0.0,1047.152,18.248,0.017
ac_measured,0.0,0.217,1.282,4.896,-11.343,0.0,1047.152,-4.949,-0.005
graph,0.0,182.583,24.479,4.896,-11.343,0.0,1047.152,200.615,0.192
graph_ratings,0.0,0.000,24.479,4.896,-11.343,0.0,1047.152,18.032,0.017


## The loading the fees saw vs the measured loading

Per trade, on the lines of its path. `share > 0.2` is how often the thermal fee applies (threshold 0.2), `share > 0.8` the congestion fee.

In [5]:
def summary(g):
    return pd.Series({
        "mean c_fee": g.c_fee.mean(), "max c_fee": g.c_fee.max(),
        "mean c_measured": g.c_measured.mean(), "max c_measured": g.c_measured.max(),
        "share > 0.2": (g.c_fee > 0.2).mean(), "share > 0.8": (g.c_fee > 0.8).mean(),
        "corr(fee, measured)": g.c_fee.corr(g.c_measured),
    })

trades.groupby("arm")[["c_fee", "c_measured"]].apply(summary).round(3)

,mean c_fee,max c_fee,mean c_measured,max c_measured,share > 0.2,share > 0.8,"corr(fee, measured)"
arm,,,,,,,
ac,0.047,0.247,0.047,0.247,0.010,0.0,1.000
ac_measured,0.047,0.247,0.047,0.247,0.010,0.0,1.000
graph,0.267,0.751,0.047,0.247,0.629,0.0,0.388
graph_ratings,0.036,0.144,0.047,0.247,0.000,0.0,0.584


## Losses: the paper's κ·d·q against the power flow

`trade_losses` is eq. 18 summed over the trades; `grid_losses` is the power flow's total (lines and transformer).

In [6]:
steps[steps.arm == "ac"][["trade_losses", "grid_losses"]].sum().round(1)

trade_losses     18.7
grid_losses     182.0
dtype: float64